ランダムネス小

In [1]:
import numpy as np
import pandas as pd
import random

class DihedralHighEntropyOptimizer:
    def __init__(self, n=384):
        self.n = n
        self.P = 2 * n # 768
        self.central = n // 2

    def get_element(self, k_a, k_b, is_ref_a=False, is_ref_b=False):
        p = list(range(self.P))
        # 領域 A の作用
        for x in range(self.n):
            if not is_ref_a: # Rotation
                p[x] = (x + k_a) % self.n
            else: # Reflection (中心 0 で反転)
                p[x] = (self.n - x + k_a) % self.n
        # 領域 B の作用
        for x in range(self.n):
            if not is_ref_b: # Rotation
                p[x + self.n] = self.n + (x + k_b) % self.n
            else: # Reflection
                p[x + self.n] = self.n + (self.n - x + k_b) % self.n
        return tuple(p)

    def solve(self):
        F = []
        G = []
        # F ブロックの構成
        for i in range(6):
            if i == 0:   # f0: Aで反転, Bで中心元
                F.append(self.get_element(0, self.central, is_ref_a=True))
            elif i == 1: # f1: Aで中心元, Bで反転
                F.append(self.get_element(self.central, 0, is_ref_b=True))
            else:        # その他: 両領域で中心元
                F.append(self.get_element(self.central, self.central))

        # G ブロックの構成
        for j in range(6):
            if j == 2:   # g2: Aで中心元, Bで一般回転 (f1と衝突)
                G.append(self.get_element(self.central, 123))
            elif j == 3: # g3: Aで一般回転 (f0と衝突), Bで中心元
                G.append(self.get_element(123, self.central))
            else:        # その他: 両領域で中心元
                G.append(self.get_element(self.central, self.central))
        
        return F, G

def check_commute(p1, p2):
    for x in range(len(p1)):
        if p1[p2[x]] != p2[p1[x]]: return 0
    return 1

# 実行
dq = DihedralHighEntropyOptimizer()
F, G = dq.solve()

# 可換表の表示
matrix = np.zeros((6, 6), dtype=int)
for i in range(6):
    for j in range(6):
        matrix[i, j] = check_commute(F[i], G[j])

df = pd.DataFrame(matrix, index=[f'f{i}' for i in range(6)], columns=[f'g{j}' for j in range(6)])
print("--- 最終修正版：混合可換表 ---")
print(df)

--- 最終修正版：混合可換表 ---
    g0  g1  g2  g3  g4  g5
f0   1   1   1   0   1   1
f1   1   1   0   1   1   1
f2   1   1   1   1   1   1
f3   1   1   1   1   1   1
f4   1   1   1   1   1   1
f5   1   1   1   1   1   1
